# DATA PREPROCESSING PIPELINE

Run all to create the train / validate / split CSVs for model training. Reads the data set in archive/, applies cleaning steps, creates deterministic train/validate/test splits, validates the split integrity, and writes the refreshed split CSVs back into data/

## Set Up

Define constants for paths, different types of features selected for use, sentinel values and missing placeholders

In [13]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# Path to raw data
RAW_INPUT_PATH = Path("../archive/data/home-credit-default-risk/application_train.csv")

# Seed to make split reproducible (ie making sure same rows go into each split every run)
RANDOM_STATE = 42
SPLIT_RATIOS = {"train": 0.70, "validate": 0.15, "test": 0.15}

# Original data sets 1000 years for days employed when missing (will replace with pd.NA)
SENTINEL_DAYS_EMPLOYED = 365243
# Placeholder to fill missing categorical values
MISSING_CATEGORY = "MISSING"


# FEATURE SELECTION (split by type of feature as we treat nulls differently for each type)

# Customer ID (keeping for now in case we want to trace specific records for diagnostics but dropped for training)
ID_COLUMN = "SK_ID_CURR"

# Label (ie credit decision)
TARGET_COLUMN = "TARGET"

# Credit Scores from Agencies
EXT_SOURCE_COLUMNS = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
EXT_SOURCE_MISSING_FLAGS = [
    "EXT_SOURCE_1_MISSING",
    "EXT_SOURCE_2_MISSING",
    "EXT_SOURCE_3_MISSING",
]

NUMERIC_FEATURES = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "OWN_CAR_AGE",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    *EXT_SOURCE_COLUMNS,
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
]

# Separated these from numercial bc original is negative # days from application date
DAY_OFFSET_COLUMNS = [
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
]

CATEGORICAL_FEATURES = [
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
]

# Grouping input features for convenience
SELECTED_COLUMNS = [
    ID_COLUMN,
    TARGET_COLUMN,
    *NUMERIC_FEATURES,
    *CATEGORICAL_FEATURES,
]

# Output after preprocessing includes new features flagging missing credit scores (could be useful to train)
OUTPUT_COLUMNS = [
    ID_COLUMN,
    TARGET_COLUMN,
    *NUMERIC_FEATURES,
    *EXT_SOURCE_MISSING_FLAGS,
    *CATEGORICAL_FEATURES,
]

# Ouput paths for the 3 splits
OUTPUT_DIR = Path("../data")
TRAIN_OUTPUT_PATH = OUTPUT_DIR / "train.csv"
VALIDATE_OUTPUT_PATH = OUTPUT_DIR / "validate.csv"
TEST_OUTPUT_PATH = OUTPUT_DIR / "test.csv"

# Info prints
print("Raw input:", RAW_INPUT_PATH)
print("Output directory:", OUTPUT_DIR)
print("Selected input columns:", len(SELECTED_COLUMNS))
print("Final output columns:", len(OUTPUT_COLUMNS))

Raw input: ../archive/data/home-credit-default-risk/application_train.csv
Output directory: ../data
Selected input columns: 30
Final output columns: 33


## Load Raw Data

Load selected features (per above) from the Home Credit dataset

In [14]:
# Load dataset (selected columns only per above)
raw_df = pd.read_csv(RAW_INPUT_PATH, usecols=SELECTED_COLUMNS, low_memory=False)    # no low mem param for better typing when loading onto df

# Show first 5 data rows for sanity check
raw_df.head()

,SK_ID_CURR,TARGET,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_INCOME_TYPE,...,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,N,Y,0,202500.0,406597.5,24700.5,351000.0,Working,...,Business Entity Type 3,0.083037,0.262949,0.139376,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,N,N,0,270000.0,1293502.5,35698.5,1129500.0,State servant,...,School,0.311267,0.622246,NaN,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Working,...,Government,NaN,0.555912,0.729567,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,N,Y,0,135000.0,312682.5,29686.5,297000.0,Working,...,Business Entity Type 3,NaN,0.650442,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,N,Y,0,121500.0,513000.0,21865.5,513000.0,Working,...,Religion,NaN,0.322738,NaN,0.0,0.0,0.0,0.0,0.0,0.0


## Raw Data EDA

Print stats re raw dataset size, label balance, duplicates and missing values

In [15]:
# Create dict of key raw-data stats
raw_summary = {
    "row_count": int(len(raw_df)),                                                          # No. of raw samples
    "column_count": int(raw_df.shape[1]),                                                   # No. of columns
    "duplicate_id_count": int(raw_df[ID_COLUMN].duplicated().sum()),                        # Duplicate applicant IDs
    "target_distribution": raw_df[TARGET_COLUMN].value_counts(dropna=False).sort_index().to_dict(),  # Raw label counts
}

# Get missing data info
raw_missing = pd.DataFrame({
    "missing_count": raw_df.isna().sum(),                        # Nulls per column
    "missing_rate_pct": (raw_df.isna().mean() * 100).round(4),  # Null rate as percent
}).sort_values("missing_rate_pct", ascending=False)              # Show most-missing columns first

# Info prints
print(raw_summary)    # Print compact dataset summary
raw_missing.head(10)  # Preview top 10 columns by missing rate

{'row_count': 307511, 'column_count': 30, 'duplicate_id_count': 0, 'target_distribution': {0: 282686, 1: 24825}}


,missing_count,missing_rate_pct
OWN_CAR_AGE,202929,65.9908
EXT_SOURCE_1,173378,56.3811
OCCUPATION_TYPE,96391,31.3455
EXT_SOURCE_3,60965,19.8253
AMT_REQ_CREDIT_BUREAU_DAY,41519,13.5016
AMT_REQ_CREDIT_BUREAU_WEEK,41519,13.5016
AMT_REQ_CREDIT_BUREAU_QRT,41519,13.5016
AMT_REQ_CREDIT_BUREAU_HOUR,41519,13.5016
AMT_REQ_CREDIT_BUREAU_MON,41519,13.5016
AMT_REQ_CREDIT_BUREAU_YEAR,41519,13.5016


## Fill Missing Values

Replace sentinel values, fill missing credit rating scores with mean of present scores when possible, fill missing categorical values with placeholder

In [16]:
clean_df = raw_df.copy()

# Original data sets 1000 years for days employed when missing (replace sentinel with pd.NA)
clean_df["DAYS_EMPLOYED"] = clean_df["DAYS_EMPLOYED"].replace(SENTINEL_DAYS_EMPLOYED, pd.NA)

# Keep track of missing credit scores before imputing
for column, flag_column in zip(EXT_SOURCE_COLUMNS, EXT_SOURCE_MISSING_FLAGS, strict=True):
    clean_df[flag_column] = clean_df[column].isna().astype(int)

# Fill each missing EXT_SOURCE value with the row-wise mean of available scores
ext_source_row_mean = clean_df[EXT_SOURCE_COLUMNS].mean(axis=1, skipna=True)
for column in EXT_SOURCE_COLUMNS:
    clean_df[column] = clean_df[column].fillna(ext_source_row_mean)

# Fill missing categorical values with explicit placeholder
for column in CATEGORICAL_FEATURES:
    clean_df[column] = clean_df[column].fillna(MISSING_CATEGORY).astype("string")

# Collect missing counts for diagnostis
ext_source_summary = {
    "missing_flag_counts": {flag: int(clean_df[flag].sum()) for flag in EXT_SOURCE_MISSING_FLAGS},
    "remaining_null_counts": {column: int(clean_df[column].isna().sum()) for column in EXT_SOURCE_COLUMNS},
}
categorical_missing_counts = {
    column: int((clean_df[column] == MISSING_CATEGORY).sum())
    for column in CATEGORICAL_FEATURES
}

print(ext_source_summary)
print(categorical_missing_counts)
clean_df[EXT_SOURCE_COLUMNS + EXT_SOURCE_MISSING_FLAGS].head()

{'missing_flag_counts': {'EXT_SOURCE_1_MISSING': 173378, 'EXT_SOURCE_2_MISSING': 660, 'EXT_SOURCE_3_MISSING': 60965}, 'remaining_null_counts': {'EXT_SOURCE_1': 172, 'EXT_SOURCE_2': 172, 'EXT_SOURCE_3': 172}}
{'FLAG_OWN_CAR': 0, 'FLAG_OWN_REALTY': 0, 'NAME_INCOME_TYPE': 0, 'NAME_EDUCATION_TYPE': 0, 'NAME_FAMILY_STATUS': 0, 'NAME_HOUSING_TYPE': 0, 'OCCUPATION_TYPE': 96391, 'ORGANIZATION_TYPE': 0}


,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,EXT_SOURCE_1_MISSING,EXT_SOURCE_2_MISSING,EXT_SOURCE_3_MISSING
0,0.083037,0.262949,0.139376,0,0,0
1,0.311267,0.622246,0.466757,0,0,1
2,0.642739,0.555912,0.729567,1,0,0
3,0.650442,0.650442,0.650442,1,0,1
4,0.322738,0.322738,0.322738,1,0,1


In [17]:
# Drop rows still missing any EXT_SOURCE value
rows_before_drop = len(clean_df)
clean_df = clean_df.dropna(subset=EXT_SOURCE_COLUMNS).copy()
rows_dropped_for_remaining_nulls = rows_before_drop - len(clean_df)

# Info prints
print("Rows dropped for remaining EXT_SOURCE nulls:", rows_dropped_for_remaining_nulls)
print("Rows remaining after null drop:", len(clean_df))

Rows dropped for remaining EXT_SOURCE nulls: 172
Rows remaining after null drop: 307339


## Clean Data

Drop unusable rows, verify dups by ID, drop invalid labels, convert day-offset fields to positive, keep only final output schema in clean df

In [18]:
# Drop if no label or id (NOTE: dropping for ID debatable since not used in training but preserves tracing per above comment)
clean_df = clean_df.dropna(subset=[ID_COLUMN, TARGET_COLUMN]).copy()

# Dedup records
duplicate_id_count = int(clean_df[ID_COLUMN].duplicated().sum())
if duplicate_id_count:
    raise ValueError(f"{ID_COLUMN} must be unique. Found {duplicate_id_count} duplicate rows.")

# Validate labels (keep only 0/1 rows)
clean_df[TARGET_COLUMN] = pd.to_numeric(clean_df[TARGET_COLUMN], errors="coerce")   # Coerce label values to numeric or Nan if error
clean_df = clean_df[clean_df[TARGET_COLUMN].isin([0, 1])].copy()                      # Filter out all rows with label other than 0/1
clean_df[TARGET_COLUMN] = clean_df[TARGET_COLUMN].astype(int)                         # Ensure dtype is int

# Make day-offset fields positive (original data counts neg num of days before application)
for column in DAY_OFFSET_COLUMNS:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce").abs()

# Keep only final output columns
clean_df = clean_df[OUTPUT_COLUMNS]

# Confirm label values
observed_targets = set(clean_df[TARGET_COLUMN].unique().tolist())
print("Observed target values:", sorted(observed_targets))
print("Rows after ID/target validation:", len(clean_df))
print("Duplicate applicant IDs:", duplicate_id_count)

# Show first 5 cleaned rows for sanity check
clean_df.head()

Observed target values: [0, 1]
Rows after ID/target validation: 307339
Duplicate applicant IDs: 0


,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,...,EXT_SOURCE_2_MISSING,EXT_SOURCE_3_MISSING,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,OCCUPATION_TYPE,ORGANIZATION_TYPE
0,100002,1,202500.0,406597.5,24700.5,351000.0,9461,637.0,3648.0,2120,...,0,0,N,Y,Working,Secondary / secondary special,Single / not married,House / apartment,Laborers,Business Entity Type 3
1,100003,0,270000.0,1293502.5,35698.5,1129500.0,16765,1188.0,1186.0,291,...,0,1,N,N,State servant,Higher education,Married,House / apartment,Core staff,School
2,100004,0,67500.0,135000.0,6750.0,135000.0,19046,225.0,4260.0,2531,...,0,0,Y,Y,Working,Secondary / secondary special,Single / not married,House / apartment,Laborers,Government
3,100006,0,135000.0,312682.5,29686.5,297000.0,19005,3039.0,9833.0,2437,...,0,1,N,Y,Working,Secondary / secondary special,Civil marriage,House / apartment,Laborers,Business Entity Type 3
4,100007,0,121500.0,513000.0,21865.5,513000.0,19932,3038.0,4311.0,3458,...,0,1,N,Y,Working,Secondary / secondary special,Single / not married,House / apartment,Core staff,Religion


## Print key info after cleaning

Print the same high-level diagnostics after cleaning so the effects of preprocessing are visible before splitting.

In [19]:
# Create dict of key cleaned-data stats
clean_summary = {
    "row_count": int(len(clean_df)),                                                           # No. of cleaned samples
    "column_count": int(clean_df.shape[1]),                                                    # No. of columns
    "duplicate_id_count": int(clean_df[ID_COLUMN].duplicated().sum()),                         # Duplicate applicant IDs
    "target_distribution": clean_df[TARGET_COLUMN].value_counts(dropna=False).sort_index().to_dict(),  # Cleaned label counts
}

# Print dtypes
# Group for dtype: [list of columns]
dtype_to_columns = (
    clean_df.dtypes
    .astype(str)                                # Convert dtype objects to readable strings
    .groupby(clean_df.dtypes.astype(str))       # Group columns by dtype
    .apply(lambda s: sorted(s.index.tolist()))  # Collect and sort column names in each dtype group
)

# Check dtypes
print("\nFeatures by dtype:")
for dtype_name, columns in dtype_to_columns.items():
    print(f"{dtype_name}: {columns}")

# Print missings
print("\nFeatures by dtype:")
clean_missing = pd.DataFrame({
    "missing_count": clean_df.isna().sum(),                        # Nulls per column
    "missing_rate_pct": (clean_df.isna().mean() * 100).round(4),  # Null rate as percent
}).sort_values("missing_rate_pct", ascending=False)                # Show most-missing columns first
print(clean_summary)    # Print compact dataset summary
clean_missing.head(10)  # Preview top 10 columns by missing rate


Features by dtype:
float64: ['AMT_ANNUITY', 'AMT_CREDIT', 'AMT_GOODS_PRICE', 'AMT_INCOME_TOTAL', 'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_YEAR', 'CNT_FAM_MEMBERS', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'OWN_CAR_AGE']
int64: ['CNT_CHILDREN', 'DAYS_BIRTH', 'DAYS_ID_PUBLISH', 'EXT_SOURCE_1_MISSING', 'EXT_SOURCE_2_MISSING', 'EXT_SOURCE_3_MISSING', 'SK_ID_CURR', 'TARGET']
string: ['FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE']

Features by dtype:
{'row_count': 307339, 'column_count': 33, 'duplicate_id_count': 0, 'target_distribution': {0: 282528, 1: 24811}}


,missing_count,missing_rate_pct
OWN_CAR_AGE,202801,65.9861
DAYS_EMPLOYED,55342,18.0068
AMT_REQ_CREDIT_BUREAU_QRT,41437,13.4825
AMT_REQ_CREDIT_BUREAU_HOUR,41437,13.4825
AMT_REQ_CREDIT_BUREAU_YEAR,41437,13.4825
AMT_REQ_CREDIT_BUREAU_WEEK,41437,13.4825
AMT_REQ_CREDIT_BUREAU_MON,41437,13.4825
AMT_REQ_CREDIT_BUREAU_DAY,41437,13.4825
AMT_GOODS_PRICE,278,0.0905
AMT_ANNUITY,12,0.0039


### (!) Big class inbalance

Also, 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY' are binary flags, should be converted from dtype=str to 1/0 dtype int 

In [20]:
# Convert binary flags with str dtypes to ints
binary_flag_columns = ["FLAG_OWN_CAR", "FLAG_OWN_REALTY"]

# Check current str values
for col in binary_flag_columns:
    print(f"{col} pre-conv dtype values: {sorted(clean_df[col].unique())}, dtype: {clean_df[col].dtype}")

# Convert each feature series value to True if 'Y' else False, cast to int
for col in binary_flag_columns:
    clean_df[col] = clean_df[col].eq("Y").astype(int)

# Confirm conversion
print()
for col in binary_flag_columns:
    print(f"{col} pre-conv dtype values: {sorted(clean_df[col].unique())}, dtype: {clean_df[col].dtype}")

FLAG_OWN_CAR pre-conv dtype values: ['N', 'Y'], dtype: string
FLAG_OWN_REALTY pre-conv dtype values: ['N', 'Y'], dtype: string

FLAG_OWN_CAR pre-conv dtype values: [np.int64(0), np.int64(1)], dtype: int64
FLAG_OWN_REALTY pre-conv dtype values: [np.int64(0), np.int64(1)], dtype: int64


## Create Splits

Split dataset to train / validate / test with stratified sampling on labels so the class balance stays stable across all three splits for now (each model architecture may have to work with this differently - TBD)

In [21]:
train_df, temp_df = train_test_split(
    clean_df,
    test_size=SPLIT_RATIOS["validate"] + SPLIT_RATIOS["test"],
    stratify=clean_df[TARGET_COLUMN],
    random_state=RANDOM_STATE,
)

validate_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df[TARGET_COLUMN],
    random_state=RANDOM_STATE,
)

# Sor values by ID and reset indices
train_df = train_df.sort_values(ID_COLUMN).reset_index(drop=True)
validate_df = validate_df.sort_values(ID_COLUMN).reset_index(drop=True)
test_df = test_df.sort_values(ID_COLUMN).reset_index(drop=True)

# Info print for shape
print(train_df.shape, validate_df.shape, test_df.shape)

(215137, 33) (46101, 33) (46101, 33)


## Validate Split

Confirm  all splits share same schema, no dups,no ID leakage across splits, default rate stays close to the overall rate

In [22]:
splits = {"train": train_df, "validate": validate_df, "test": test_df}
expected_columns = list(clean_df.columns)

split_checks = {
    "schema_match": {name: list(df.columns) == expected_columns for name, df in splits.items()},
    "duplicate_id_count": {name: int(df[ID_COLUMN].duplicated().sum()) for name, df in splits.items()},
    "row_count": {name: int(len(df)) for name, df in splits.items()},
    "default_rate": {name: float(df[TARGET_COLUMN].mean()) for name, df in splits.items()},
}

split_id_sets = {name: set(df[ID_COLUMN].tolist()) for name, df in splits.items()}
split_checks["id_overlap"] = {
    "train_validate": int(len(split_id_sets["train"].intersection(split_id_sets["validate"]))),
    "train_test": int(len(split_id_sets["train"].intersection(split_id_sets["test"]))),
    "validate_test": int(len(split_id_sets["validate"].intersection(split_id_sets["test"]))),
}

# Raise errors to make us aware of validation issues
if not all(split_checks["schema_match"].values()):
    raise ValueError("Split schema validation failed.")
if any(count > 0 for count in split_checks["duplicate_id_count"].values()):
    raise ValueError("Duplicate applicant IDs found within a split.")
if any(count > 0 for count in split_checks["id_overlap"].values()):
    raise ValueError("Applicant ID overlap detected across splits.")

split_checks

{'schema_match': {'train': True, 'validate': True, 'test': True},
 'duplicate_id_count': {'train': 0, 'validate': 0, 'test': 0},
 'row_count': {'train': 215137, 'validate': 46101, 'test': 46101},
 'default_rate': {'train': 0.08072995347150885,
  'validate': 0.08073577579662046,
  'test': 0.08071408429318236},
 'id_overlap': {'train_validate': 0, 'train_test': 0, 'validate_test': 0}}

## Write Split CSV Files

Splits stored in data/ directory (do not include in .gitignore so we all have the same - original raw dataset is ignored)

In [23]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(TRAIN_OUTPUT_PATH, index=False)
validate_df.to_csv(VALIDATE_OUTPUT_PATH, index=False)
test_df.to_csv(TEST_OUTPUT_PATH, index=False)

print("Wrote:")
print(TRAIN_OUTPUT_PATH)
print(VALIDATE_OUTPUT_PATH)
print(TEST_OUTPUT_PATH)

Wrote:
../data/train.csv
../data/validate.csv
../data/test.csv


## Output Summary

Print split sizes, label rates, preview resulting training data

In [24]:
full_default_rate = float(clean_df[TARGET_COLUMN].mean())

print("Full cleaned default rate:", full_default_rate)
print("Train default rate:", float(train_df[TARGET_COLUMN].mean()))
print("Validate default rate:", float(validate_df[TARGET_COLUMN].mean()))
print("Test default rate:", float(test_df[TARGET_COLUMN].mean()))

print("Train shape:", train_df.shape)
print("Validate shape:", validate_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Full cleaned default rate: 0.08072844643862315
Train default rate: 0.08072995347150885
Validate default rate: 0.08073577579662046
Test default rate: 0.08071408429318236
Train shape: (215137, 33)
Validate shape: (46101, 33)
Test shape: (46101, 33)


,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,...,EXT_SOURCE_2_MISSING,EXT_SOURCE_3_MISSING,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,OCCUPATION_TYPE,ORGANIZATION_TYPE
0,100002,1,202500.0,406597.5,24700.5,351000.0,9461,637.0,3648.0,2120,...,0,0,0,1,Working,Secondary / secondary special,Single / not married,House / apartment,Laborers,Business Entity Type 3
1,100003,0,270000.0,1293502.5,35698.5,1129500.0,16765,1188.0,1186.0,291,...,0,1,0,0,State servant,Higher education,Married,House / apartment,Core staff,School
2,100004,0,67500.0,135000.0,6750.0,135000.0,19046,225.0,4260.0,2531,...,0,0,1,1,Working,Secondary / secondary special,Single / not married,House / apartment,Laborers,Government
3,100007,0,121500.0,513000.0,21865.5,513000.0,19932,3038.0,4311.0,3458,...,0,1,0,1,Working,Secondary / secondary special,Single / not married,House / apartment,Core staff,Religion
4,100008,0,99000.0,490495.5,27517.5,454500.0,16941,1588.0,4970.0,477,...,0,0,0,1,State servant,Secondary / secondary special,Married,House / apartment,Laborers,Other
